# Training model kemacetan (fine-tune) v2
Di Google Drive `MyDrive/cctv/` harus ada `best.pt` dan `dataset_final` (folder atau `dataset_final.zip`). Runtime: **GPU** (Runtime > Change runtime type).

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics

In [ ]:
import os, shutil, glob
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/cctv'
DS = '/content/dataset_final'

shutil.copy(f'{BASE}/best.pt', '/content/best.pt')
# dataset disalin ke disk lokal Colab (jauh lebih cepat daripada membaca langsung dari Drive)
if os.path.isdir(f'{BASE}/dataset_final'):
    if not os.path.isdir(DS):
        shutil.copytree(f'{BASE}/dataset_final', DS)
elif os.path.exists(f'{BASE}/dataset_final.zip'):
    !unzip -q -o {BASE}/dataset_final.zip -d /content
else:
    raise SystemExit('dataset_final tidak ditemukan di Drive')

# pemeriksaan: harus 2515 train, 102 val, 72 test
for s in ('train', 'val', 'test'):
    print(s, len(glob.glob(f'{DS}/images/{s}/*')), 'gambar', len(glob.glob(f'{DS}/labels/{s}/*.txt')), 'label')
for f in ('data.yaml', 'bogor.yaml', 'val_bogor.txt'):
    assert os.path.exists(f'{DS}/{f}'), f + ' tidak ada'
print(open(f'{DS}/data.yaml').read())

## Training
Hasil disimpan ke Drive supaya tidak hilang jika sesi terputus.

In [ ]:
from ultralytics import YOLO
model = YOLO('/content/best.pt')
model.train(
    data=f'{DS}/data.yaml',
    imgsz=640, epochs=60, patience=15, batch=16, workers=2,
    project=f'{BASE}/runs', name='v2', exist_ok=True,
)

## Evaluasi di frame Bogor (15 gambar)
CATATAN: label Bogor belum dikoreksi manual (berasal dari model lama), jadi skor ini hanya indikasi kasar, bukan bukti akurasi.

In [ ]:
RUN = f'{BASE}/runs/v2'
for name, w in [('LAMA', '/content/best.pt'), ('BARU', RUN + '/weights/best.pt')]:
    m = YOLO(w).val(data=f'{DS}/bogor.yaml', imgsz=640, plots=False)
    print(name, 'recall=%.3f mAP50=%.3f mAP50-95=%.3f' % (m.box.mr, m.box.map50, m.box.map))

## Ekspor untuk worker (OpenVINO, ukuran 640)

In [ ]:
best = YOLO(RUN + '/weights/best.pt')
best.export(format='openvino', imgsz=640)
!cp -r {RUN}/weights/best_openvino_model {BASE}/best_openvino_model_v2
!ls {BASE}